# Querying PGx Data

In [ ]:
#load environmental variables and install packages if not already loaded
%run -i ./00_setup_workspace_env.py

In [1]:
try:
    import polars as pl
except ImportError:  # install only if the image does not already have it
    get_ipython().run_line_magic("pip", "install -q polars")
    import polars as pl

import gcsfs

# PGx result files are read directly from the All of Us bucket (requester pays),
# so no workspace mount path is needed.
fs = gcsfs.GCSFileSystem(requester_pays=True)
pgx_bucket = "vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/pgx"
high_conf_prefix = f"{pgx_bucket}/high_concordance/"
low_conf_prefix = f"{pgx_bucket}/low_concordance/"

def read_pgx(path, **kwargs):
    """Read a tab-separated PGx results file from the bucket into a Polars DataFrame."""
    with fs.open(path) as f:
        return pl.read_csv(f, **kwargs)

def mask_small(df, count_col="count", threshold=20):
    """Return df with counts of 1..threshold replaced by "≤<threshold>".

    The All of Us Data and Statistics Dissemination Policy prohibits reporting
    participant counts of 1-20, so every table printed in this notebook goes
    through this function. Use count_col="len" for tables made with pl.len().
    """
    return df.with_columns(
        pl.when(pl.col(count_col).is_between(1, threshold))
        .then(pl.lit(f"≤{threshold}"))
        .otherwise(pl.col(count_col).cast(pl.String))
        .alias(count_col)
    )

Workspace environment set (GOOGLE_CLOUD_PROJECT, WORKSPACE_BUCKET, WORKSPACE_CDR)
  CDR: C2025Q4R6
Run record:
  Notebook start: 2026-11-13T21:40:55+00:00
  Environment: Standard (Jupyter)
  Python: 3.12.16
  CPUs: 4
  Memory (GB, 'free -g' Mem row): Mem:              25           2          18           0           5          23
  Disk (/home): overlay         492G   42G  429G   9% /


# CYP2C_Cluster

In [2]:
# Read the TSV file into a Polars DataFrame
CYP2C_df = read_pgx(f"{high_conf_prefix}2c_cluster_cohort_results.tsv", separator="\t")

In [3]:
# Group by 'gene' and 'genotype' and count the occurrences
grouped_counts = CYP2C_df.group_by(["gene", "genotype", "phenotype"]).agg(pl.len())

# Print the entire Polars DataFrame
pl.Config.set_tbl_rows(-1)  # show all rows
print(mask_small(grouped_counts, "len"))

shape: (3, 4)
┌────────────┬─────────────────────────────────┬─────────────────┬────────┐
│ gene       ┆ genotype                        ┆ phenotype       ┆ len    │
│ ---        ┆ ---                             ┆ ---             ┆ ---    │
│ str        ┆ str                             ┆ str             ┆ str    │
╞════════════╪═════════════════════════════════╪═════════════════╪════════╡
│ 2C_CLUSTER ┆ rs12777823 reference (G)/rs127… ┆ Variant Absent  ┆ 369248 │
│ 2C_CLUSTER ┆ rs12777823 reference (G)/rs127… ┆ Variant Present ┆ 149070 │
│ 2C_CLUSTER ┆ rs12777823 variant (A)/rs12777… ┆ Variant Present ┆ 17314  │
└────────────┴─────────────────────────────────┴─────────────────┴────────┘


# ABCG2

In [4]:
# Read the TSV file into a Polars DataFrame
ABCG2_df = read_pgx(f"{high_conf_prefix}abcg2_cohort_results.tsv", separator="\t")

In [5]:
# Group by 'gene' and 'genotype' and count the occurrences
grouped_counts = ABCG2_df.group_by(["gene", "genotype", "phenotype"]).agg(pl.len())

# Print the entire Polars DataFrame
pl.Config.set_tbl_rows(-1)  # show all rows
print(mask_small(grouped_counts, "len"))

shape: (3, 4)
┌───────┬─────────────────────────────────┬────────────────────┬────────┐
│ gene  ┆ genotype                        ┆ phenotype          ┆ len    │
│ ---   ┆ ---                             ┆ ---                ┆ ---    │
│ str   ┆ str                             ┆ str                ┆ str    │
╞═══════╪═════════════════════════════════╪════════════════════╪════════╡
│ ABCG2 ┆ rs2231142 reference (G)/rs2231… ┆ Normal Function    ┆ 427044 │
│ ABCG2 ┆ rs2231142 reference (G)/rs2231… ┆ Decreased Function ┆ 100088 │
│ ABCG2 ┆ rs2231142 variant (T)/rs223114… ┆ Poor Function      ┆ 8500   │
└───────┴─────────────────────────────────┴────────────────────┴────────┘


# CACNA1S

In [6]:
CACNA1S_df = read_pgx(
    f"{high_conf_prefix}cacna1s_cohort_results.tsv",
    separator="\t",
    null_values=["n/a"]
)

In [7]:
# Group by 'gene' and 'genotype' and count the occurrences
grouped_counts = CACNA1S_df.group_by(["gene", "genotype", "phenotype"]).agg(pl.len())

# Print the entire Polars DataFrame
pl.Config.set_tbl_rows(-1) # Don't print counts <20

polars.config.Config

In [8]:
phenotype_counts = (
    CACNA1S_df
    .group_by("phenotype")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

print(mask_small(phenotype_counts))

shape: (2, 2)
┌─────────────────────────────────┬────────┐
│ phenotype                       ┆ count  │
│ ---                             ┆ ---    │
│ str                             ┆ str    │
╞═════════════════════════════════╪════════╡
│ Uncertain Susceptibility        ┆ 535597 │
│ Malignant Hyperthermia Suscept… ┆ 35     │
└─────────────────────────────────┴────────┘


# CFTR

In [9]:
# Read the TSV file into a Polars DataFrame
CFTR_df = read_pgx(f"{high_conf_prefix}cftr_cohort_results.tsv", separator="\t")

In [10]:
# Group by 'gene' and 'genotype' and count the occurrences
grouped_counts = CFTR_df.group_by(["gene", "genotype", "phenotype"]).agg(pl.len())

# Print the entire Polars DataFrame
pl.Config.set_tbl_rows(-1)  # Don't print counts <20

polars.config.Config

In [11]:
phenotype_counts = (
    CFTR_df
    .group_by("phenotype")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

print(mask_small(phenotype_counts))

shape: (3, 2)
┌────────────────────┬────────┐
│ phenotype          ┆ count  │
│ ---                ┆ ---    │
│ str                ┆ str    │
╞════════════════════╪════════╡
│ normal_function    ┆ 526608 │
│ unknown_function   ┆ 5554   │
│ decreased_function ┆ 3470   │
└────────────────────┴────────┘


# CYP2D6

In [12]:
# Read the TSV file into a Polars DataFrame
CYP2D6_df = read_pgx(f"{high_conf_prefix}cyp2d6_cohort_foxtrot_results.tsv", separator="\t")

In [13]:
# Group by 'gene' and 'genotype' and count the occurrences
grouped_counts = CYP2D6_df.group_by(["gene", "genotype", "phenotype"]).agg(pl.len())

# Print the entire Polars DataFrame
pl.Config.set_tbl_rows(-1)  # Don't print counts <20

polars.config.Config

In [14]:
phenotype_counts = (
    CYP2D6_df
    .group_by("phenotype")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

print(mask_small(phenotype_counts))

shape: (5, 2)
┌─────────────────────────────────┬────────┐
│ phenotype                       ┆ count  │
│ ---                             ┆ ---    │
│ str                             ┆ str    │
╞═════════════════════════════════╪════════╡
│ CYP2D6 Normal Metabolizer       ┆ 279865 │
│ CYP2D6 Intermediate Metabolize… ┆ 187682 │
│ CYP2D6 Indeterminate Metaboliz… ┆ 27802  │
│ CYP2D6 Poor Metabolizer         ┆ 26002  │
│ CYP2D6 Ultrarapid Metabolizer   ┆ 14305  │
└─────────────────────────────────┴────────┘


# G6PD

In [15]:
# Read the TSV file into a Polars DataFrame
G6PD_df = read_pgx(f"{high_conf_prefix}g6pd_cohort_results.tsv", separator="\t")

In [16]:
# Group by 'gene' and 'genotype' and count the occurrences
grouped_counts = G6PD_df.group_by(["gene", "genotype", "phenotype"]).agg(pl.len())

# Print the entire Polars DataFrame
pl.Config.set_tbl_rows(-1)  # Don't print counts <20

polars.config.Config

In [17]:
phenotype_counts = (
    G6PD_df
    .group_by("phenotype")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

print(mask_small(phenotype_counts))

shape: (4, 2)
┌───────────────┬────────┐
│ phenotype     ┆ count  │
│ ---           ┆ ---    │
│ str           ┆ str    │
╞═══════════════╪════════╡
│ Normal        ┆ 510345 │
│ Variable      ┆ 23680  │
│ Deficient     ┆ 1070   │
│ Indeterminate ┆ 537    │
└───────────────┴────────┘


# NUDT15

In [18]:
# Read the TSV file into a Polars DataFrame
NUDT15_df = read_pgx(f"{high_conf_prefix}nudt15_cohort_results.tsv", separator="\t")

In [19]:
# Group by 'gene' and 'genotype' and count the occurrences
grouped_counts = NUDT15_df.group_by(["gene", "genotype", "phenotype"]).agg(pl.len())

# Print the entire Polars DataFrame
pl.Config.set_tbl_rows(-1)  # Don't print counts <20

polars.config.Config

In [20]:
phenotype_counts = (
    NUDT15_df
    .group_by("phenotype")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

print(mask_small(phenotype_counts))

shape: (5, 2)
┌─────────────────────────────────┬────────┐
│ phenotype                       ┆ count  │
│ ---                             ┆ ---    │
│ str                             ┆ str    │
╞═════════════════════════════════╪════════╡
│ Normal Metabolizer              ┆ 514558 │
│ Intermediate Metabolizer        ┆ 14521  │
│ Indeterminate                   ┆ 5943   │
│ Poor Metabolizer                ┆ 432    │
│ Possible Intermediate Metaboli… ┆ 178    │
└─────────────────────────────────┴────────┘


# RYR1

In [21]:
# Read the TSV file into a Polars DataFrame
RYR1_df = read_pgx(f"{high_conf_prefix}ryr1_cohort_results.tsv", separator="\t")

In [22]:
# Group by 'gene' and 'genotype' and count the occurrences
grouped_counts = RYR1_df.group_by(["gene", "genotype", "phenotype"]).agg(pl.len())

# Print the entire Polars DataFrame
pl.Config.set_tbl_rows(-1)  # Don't print counts <20

polars.config.Config

In [23]:
phenotype_counts = (
    RYR1_df
    .group_by("phenotype")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

print(mask_small(phenotype_counts)) #Don't print counts <20

shape: (3, 2)
┌─────────────────────────────────┬────────┐
│ phenotype                       ┆ count  │
│ ---                             ┆ ---    │
│ str                             ┆ str    │
╞═════════════════════════════════╪════════╡
│ Uncertain Susceptibility        ┆ 535210 │
│ Malignant Hyperthermia Suscept… ┆ 408    │
│ Indeterminate                   ┆ ≤20    │
└─────────────────────────────────┴────────┘


# TPMT

In [24]:
# Read the TSV file into a Polars DataFrame
TPMT_df = read_pgx(f"{high_conf_prefix}tpmt_cohort_results.tsv", separator="\t")

In [25]:
# Group by 'gene' and 'genotype' and count the occurrences
grouped_counts = TPMT_df.group_by(["gene", "genotype", "phenotype"]).agg(pl.len())

# Print the entire Polars DataFrame
pl.Config.set_tbl_rows(-1)  # Don't print counts <20

polars.config.Config

In [26]:
phenotype_counts = (
    TPMT_df
    .group_by("phenotype")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

print(mask_small(phenotype_counts))

shape: (5, 2)
┌─────────────────────────────────┬────────┐
│ phenotype                       ┆ count  │
│ ---                             ┆ ---    │
│ str                             ┆ str    │
╞═════════════════════════════════╪════════╡
│ Normal Metabolizer              ┆ 475123 │
│ Intermediate Metabolizer        ┆ 47130  │
│ Indeterminate                   ┆ 11718  │
│ Poor Metabolizer                ┆ 1017   │
│ Possible Intermediate Metaboli… ┆ 644    │
└─────────────────────────────────┴────────┘


# VKORC1

In [27]:
# Read the TSV file into a Polars DataFrame
VKORC1_df = read_pgx(f"{high_conf_prefix}vkorc1_cohort_results.tsv", separator="\t")

In [28]:
# Group by 'gene' and 'genotype' and count the occurrences
grouped_counts = VKORC1_df.group_by(["gene", "genotype", "phenotype"]).agg(pl.len())

# Print the entire Polars DataFrame
pl.Config.set_tbl_rows(-1)  # Don't print counts <20
print(mask_small(grouped_counts, "len"))

shape: (3, 4)
┌────────┬─────────────────────────────────┬─────────────────┬────────┐
│ gene   ┆ genotype                        ┆ phenotype       ┆ len    │
│ ---    ┆ ---                             ┆ ---             ┆ ---    │
│ str    ┆ str                             ┆ str             ┆ str    │
╞════════╪═════════════════════════════════╪═════════════════╪════════╡
│ VKORC1 ┆ rs9923231 reference (C)/rs9923… ┆ Variant Present ┆ 220426 │
│ VKORC1 ┆ rs9923231 reference (C)/rs9923… ┆ Variant Absent  ┆ 236932 │
│ VKORC1 ┆ rs9923231 variant (T)/rs992323… ┆ Variant Present ┆ 78272  │
└────────┴─────────────────────────────────┴─────────────────┴────────┘


In [29]:
phenotype_counts = (
    VKORC1_df
    .group_by("phenotype")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

print(mask_small(phenotype_counts)) #Note that variant present includes both hom and het counts

shape: (2, 2)
┌─────────────────┬────────┐
│ phenotype       ┆ count  │
│ ---             ┆ ---    │
│ str             ┆ str    │
╞═════════════════╪════════╡
│ Variant Present ┆ 298698 │
│ Variant Absent  ┆ 236932 │
└─────────────────┴────────┘


# CYP2B6

In [30]:
# Read the TSV file into a Polars DataFrame
CYP2B6_df = read_pgx(f"{high_conf_prefix}cyp2b6_cohort_results.tsv", separator="\t")

In [31]:
# Group by 'gene' and 'genotype' and count the occurrences
grouped_counts = CYP2B6_df.group_by(["gene", "genotype", "phenotype"]).agg(pl.len())

# Print the entire Polars DataFrame
pl.Config.set_tbl_rows(-1)  # Don't print counts <20

polars.config.Config

In [32]:
phenotype_counts = (
    CYP2B6_df
    .group_by("phenotype")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

print(mask_small(phenotype_counts))

shape: (5, 2)
┌──────────────────────────┬────────┐
│ phenotype                ┆ count  │
│ ---                      ┆ ---    │
│ str                      ┆ str    │
╞══════════════════════════╪════════╡
│ normal_metabolizer       ┆ 437184 │
│ intermediate_metabolizer ┆ 61053  │
│ ultrarapid_metabolizer   ┆ 21657  │
│ unknown_metabolizer      ┆ 15196  │
│ poor_metabolizer         ┆ 542    │
└──────────────────────────┴────────┘


# CYP2C19

In [33]:
# Read the TSV file into a Polars DataFrame
CYP2C19_df = read_pgx(f"{low_conf_prefix}cyp2c19_cohort_results.tsv", separator="\t")

In [34]:
# Group by 'gene' and 'genotype' and count the occurrences
grouped_counts = CYP2C19_df.group_by(["gene", "genotype", "phenotype"]).agg(pl.len())

# Print the entire Polars DataFrame
pl.Config.set_tbl_rows(-1)  # Don't print counts <20

polars.config.Config

In [35]:
phenotype_counts = (
    CYP2C19_df
    .group_by("phenotype")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

print(mask_small(phenotype_counts))

shape: (8, 2)
┌─────────────────────────────────┬────────┐
│ phenotype                       ┆ count  │
│ ---                             ┆ ---    │
│ str                             ┆ str    │
╞═════════════════════════════════╪════════╡
│ Normal Metabolizer              ┆ 217824 │
│ Intermediate Metabolizer        ┆ 142059 │
│ Rapid Metabolizer               ┆ 128163 │
│ Ultrarapid Metabolizer          ┆ 21707  │
│ Poor Metabolizer                ┆ 15261  │
│ Indeterminate                   ┆ 6802   │
│ Likely Intermediate Metabolize… ┆ 3052   │
│ Likely Poor Metabolizer         ┆ 764    │
└─────────────────────────────────┴────────┘


# CYP3A5

In [36]:
# Read the TSV file into a Polars DataFrame
CYP3A5_df = read_pgx(f"{high_conf_prefix}cyp3a5_cohort_results.tsv", separator="\t")

In [37]:
# Group by 'gene' and 'genotype' and count the occurrences
grouped_counts = CYP3A5_df.group_by(["gene", "genotype", "phenotype"]).agg(pl.len())

# Print the entire Polars DataFrame
pl.Config.set_tbl_rows(-1)  # Don't print counts <20

polars.config.Config

In [38]:
phenotype_counts = (
    CYP3A5_df
    .group_by("phenotype")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

print(mask_small(phenotype_counts))

shape: (5, 2)
┌─────────────────────────────────┬────────┐
│ phenotype                       ┆ count  │
│ ---                             ┆ ---    │
│ str                             ┆ str    │
╞═════════════════════════════════╪════════╡
│ Poor Metabolizer                ┆ 376957 │
│ Intermediate Metabolizer        ┆ 129235 │
│ Normal Metabolizer              ┆ 29377  │
│ Indeterminate                   ┆ 40     │
│ Possible Intermediate Metaboli… ┆ ≤20    │
└─────────────────────────────────┴────────┘


# CYP4F2

In [39]:
# Read the TSV file into a Polars DataFrame
CYP4F2_df = read_pgx(f"{high_conf_prefix}cyp4f2_cohort_results.tsv", separator="\t")

In [40]:
# Group by 'gene' and 'genotype' and count the occurrences
grouped_counts = CYP4F2_df.group_by(["gene", "genotype", "phenotype"]).agg(pl.len())

# Print the entire Polars DataFrame
pl.Config.set_tbl_rows(-1)  # show all rows
print(mask_small(grouped_counts, "len"))

shape: (7, 4)
┌────────┬─────────────────────────────┬────────────────────────┬────────┐
│ gene   ┆ genotype                    ┆ phenotype              ┆ len    │
│ ---    ┆ ---                         ┆ ---                    ┆ ---    │
│ str    ┆ str                         ┆ str                    ┆ str    │
╞════════╪═════════════════════════════╪════════════════════════╪════════╡
│ CYP4F2 ┆ *1/*1                       ┆ CYP4F2 Variant Absent  ┆ 273009 │
│ CYP4F2 ┆ *3/*3                       ┆ CYP4F2 Variant Present ┆ 7537   │
│ CYP4F2 ┆ *1/*3                       ┆ CYP4F2 Variant Present ┆ 83344  │
│ CYP4F2 ┆ *1/*2                       ┆ CYP4F2 Variant Absent  ┆ 31239  │
│ CYP4F2 ┆ *3/*2                       ┆ CYP4F2 Variant Present ┆ 105068 │
│ CYP4F2 ┆ Indeterminate/Indeterminate ┆ CYP4F2 Indeterminate   ┆ 31986  │
│ CYP4F2 ┆ *2/*2                       ┆ CYP4F2 Variant Absent  ┆ 3449   │
└────────┴─────────────────────────────┴────────────────────────┴────────┘


In [41]:
phenotype_counts = (
    CYP4F2_df
    .group_by("phenotype")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

print(mask_small(phenotype_counts))

shape: (3, 2)
┌────────────────────────┬────────┐
│ phenotype              ┆ count  │
│ ---                    ┆ ---    │
│ str                    ┆ str    │
╞════════════════════════╪════════╡
│ CYP4F2 Variant Absent  ┆ 307697 │
│ CYP4F2 Variant Present ┆ 195949 │
│ CYP4F2 Indeterminate   ┆ 31986  │
└────────────────────────┴────────┘


# DPYD

In [42]:
# Read the TSV file into a Polars DataFrame
DPYD_df = read_pgx(f"{low_conf_prefix}dpyd_cohort_results.tsv", separator="\t")

In [43]:
# Group by 'gene' and 'genotype' and count the occurrences
grouped_counts = DPYD_df.group_by(["gene", "genotype", "phenotype"]).agg(pl.len())

# Print the entire Polars DataFrame
pl.Config.set_tbl_rows(-1)  # Don't print counts <20

polars.config.Config

In [44]:
phenotype_counts = (
    DPYD_df
    .group_by("phenotype")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

print(mask_small(phenotype_counts))

shape: (4, 2)
┌───────────────────────────────┬────────┐
│ phenotype                     ┆ count  │
│ ---                           ┆ ---    │
│ str                           ┆ str    │
╞═══════════════════════════════╪════════╡
│ DPYD Normal Metabolizer       ┆ 507955 │
│ DPYD Intermediate Metabolizer ┆ 27162  │
│ Indeterminate                 ┆ 400    │
│ DPYD Poor Metabolizer         ┆ 113    │
└───────────────────────────────┴────────┘


# SLCO1B1

In [45]:
# Read the TSV file into a Polars DataFrame
SLCO1B1_df = read_pgx(f"{low_conf_prefix}slco1b1_cohort_results.tsv", separator="\t")

In [46]:
# Group by 'gene' and 'genotype' and count the occurrences
grouped_counts = SLCO1B1_df.group_by(["gene", "genotype", "phenotype"]).agg(pl.len())

# Print the entire Polars DataFrame
pl.Config.set_tbl_rows(-1)  # Don't print counts <20

polars.config.Config

In [47]:
phenotype_counts = (
    SLCO1B1_df
    .group_by("phenotype")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

print(mask_small(phenotype_counts))

shape: (6, 2)
┌─────────────────────────────┬────────┐
│ phenotype                   ┆ count  │
│ ---                         ┆ ---    │
│ str                         ┆ str    │
╞═════════════════════════════╪════════╡
│ Normal Function             ┆ 359355 │
│ Decreased Function          ┆ 115342 │
│ Indeterminate               ┆ 31268  │
│ Increased Function          ┆ 17076  │
│ Poor Function               ┆ 10531  │
│ Possible Decreased Function ┆ 2059   │
└─────────────────────────────┴────────┘


# UGT1A1

In [48]:
# Read the TSV file into a Polars DataFrame
UGT1A1_df = read_pgx(f"{high_conf_prefix}ugt1a1_cohort_results.tsv", separator="\t")

In [49]:
# Group by 'gene' and 'genotype' and count the occurrences
grouped_counts = UGT1A1_df.group_by(["gene", "genotype", "phenotype"]).agg(pl.len())

# Print the entire Polars DataFrame
pl.Config.set_tbl_rows(-1)  # Don't print counts <20

polars.config.Config

In [50]:
phenotype_counts = (
    UGT1A1_df
    .group_by("phenotype")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

print(mask_small(phenotype_counts))

shape: (4, 2)
┌──────────────────────────┬────────┐
│ phenotype                ┆ count  │
│ ---                      ┆ ---    │
│ str                      ┆ str    │
╞══════════════════════════╪════════╡
│ Intermediate Metabolizer ┆ 242578 │
│ Normal Metabolizer       ┆ 222365 │
│ Poor Metabolizer         ┆ 70307  │
│ Indeterminate            ┆ 382    │
└──────────────────────────┴────────┘


# CYP2C9

In [51]:
# Read the TSV file into a Polars DataFrame
CYP2C9_df = read_pgx(f"{high_conf_prefix}cyp2c9_cohort_results.tsv", separator="\t", null_values=["n/a", "NA", ""],)

In [52]:
# Group by 'gene' and 'genotype' and count the occurrences
grouped_counts = CYP2C9_df.group_by(["gene", "genotype", "phenotype"]).agg(pl.len())

# Print the entire Polars DataFrame
pl.Config.set_tbl_rows(-1)  # Don't print counts <20

polars.config.Config

In [53]:
phenotype_counts = (
    CYP2C9_df
    .group_by("phenotype")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

print(mask_small(phenotype_counts))

shape: (4, 2)
┌──────────────────────────┬────────┐
│ phenotype                ┆ count  │
│ ---                      ┆ ---    │
│ str                      ┆ str    │
╞══════════════════════════╪════════╡
│ Normal Metabolizer       ┆ 365867 │
│ Intermediate Metabolizer ┆ 159039 │
│ Poor Metabolizer         ┆ 9202   │
│ Indeterminate            ┆ 1524   │
└──────────────────────────┴────────┘


In [54]:
%run -i ./00.1_run_summary.py

Run summary
  CDR: C2025Q4R6
  Python: 3.12.16
  gcsfs: 2026.10.1
  polars: 2.1.0
  Notebook start: 2026-11-13T21:40:55+00:00
  Notebook end:   2026-11-13T21:41:27+00:00
  Wall-clock runtime: 0.01 h (0.5 min)
